# Vector Search index
A Delta Sync index on `docs_chunks` with embeddings computed by Databricks (`databricks-gte-large-en`). The endpoint is a bundle resource of the `prod` target, and `dev` shares it because Free Edition allows one endpoint. The index is created here because it needs the source table to exist. Later runs trigger a sync, which embeds only the rows changed since the last sync, and the notebook waits for that sync's pipeline update to complete.

Access: readers get `USE SCHEMA` (bundle) and `SELECT` on the chunks and the index, nothing on the raw volume or on the logs.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("src_path", "")

import sys

sys.path.insert(0, dbutils.widgets.get("src_path"))
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}"

dbutils.widgets.text("vs_endpoint", "")
dbutils.widgets.text("embedding_endpoint", "")
dbutils.widgets.text("uc_readers", "")

import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    EndpointStatusState,
    PipelineType,
    VectorIndexType,
)

w = WorkspaceClient()
vs_endpoint = dbutils.widgets.get("vs_endpoint")
source_table = f"{target}.docs_chunks"
index_name = f"{target}.docs_chunks_index"

In [ ]:
deadline = time.time() + 1800
while w.vector_search_endpoints.get_endpoint(vs_endpoint).endpoint_status.state != EndpointStatusState.ONLINE:
    if time.time() > deadline:
        raise TimeoutError(f"endpoint {vs_endpoint} is not online")
    time.sleep(30)

from databricks.sdk.service.pipelines import UpdateStateInfoState

FINISHED = (UpdateStateInfoState.COMPLETED, UpdateStateInfoState.FAILED, UpdateStateInfoState.CANCELED)


def latest_update(index):
    updates = w.pipelines.get(index.delta_sync_index_spec.pipeline_id).latest_updates or []
    return updates[0] if updates else None


previous_update = None
try:
    index = w.vector_search_indexes.get_index(index_name)
    previous = latest_update(index)
    while previous and previous.state not in FINISHED:
        if time.time() > deadline:
            raise TimeoutError("the previous index sync is still running")
        time.sleep(30)
        previous = latest_update(index)
    previous_update = previous.update_id if previous else None
    w.vector_search_indexes.sync_index(index_name)
    print("sync started")
except NotFound:
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=vs_endpoint,
        primary_key="chunk_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=source_table,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(name="content", embedding_model_endpoint_name=dbutils.widgets.get("embedding_endpoint"))
            ],
            columns_to_sync=["chunk_id", "chunk_profile", "doc_path", "doc_title", "section", "content", "source_url"],
        ),
    )
    print("index created")

In [ ]:
expected = spark.table(source_table).count()
deadline = time.time() + 3600
while True:
    index = w.vector_search_indexes.get_index(index_name)
    update = latest_update(index) if index.delta_sync_index_spec.pipeline_id else None
    if update and update.update_id != previous_update:
        if update.state in (UpdateStateInfoState.FAILED, UpdateStateInfoState.CANCELED):
            raise RuntimeError(f"index sync {update.update_id} ended {update.state.value}: {index.status.message}")
        if update.state == UpdateStateInfoState.COMPLETED and index.status.ready and index.status.indexed_row_count == expected:
            break
    if time.time() > deadline:
        raise TimeoutError(f"index not synced: {index.status.message}")
    time.sleep(30)
print(f"{index.status.indexed_row_count} rows indexed by update {update.update_id}")

In [ ]:
readers = dbutils.widgets.get("uc_readers")
spark.sql(f"GRANT SELECT ON TABLE {source_table} TO `{readers}`")
spark.sql(f"GRANT SELECT ON TABLE {index_name} TO `{readers}`")
display(spark.sql(f"SHOW GRANTS ON TABLE {index_name}"))

from rag_kb.service import LOG_COLUMNS

request_id, rest = LOG_COLUMNS.split(", ", 1)
spark.sql(f"CREATE TABLE IF NOT EXISTS {target}.qa_log ({request_id}, logged_at timestamp, user_name string, {rest})")